# 🧠 MNIST CNN Training
**Features:** CNN Architecture · Early Stopping · Model Checkpointing · PyTorch Built-in Dataset

## 1. Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import os, time

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Hyperparameters

In [ ]:
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-3
NUM_EPOCHS      = 50
VAL_SPLIT       = 0.1

# Early Stopping
ES_PATIENCE     = 5
ES_MIN_DELTA    = 1e-4

# Checkpoint
CHECKPOINT_DIR  = 'checkpoints'
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Hyperparameters loaded ✅')

## 3. Dataset & DataLoaders
> Using `torchvision.datasets.MNIST` — downloads automatically on first run.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=train_transform)
test_dataset       = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

val_size   = int(len(full_train_dataset) * VAL_SPLIT)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {train_size:,} | Val: {val_size:,} | Test: {len(test_dataset):,}')

### 3.1 Visualise a batch

In [ ]:
def show_batch(loader, n=16):
    images, labels = next(iter(loader))
    images, labels = images[:n], labels[:n]
    fig, axes = plt.subplots(2, n // 2, figsize=(14, 4))
    for i, ax in enumerate(axes.flat):
        img = images[i].squeeze().numpy() * 0.3081 + 0.1307
        ax.imshow(img, cmap='gray')
        ax.set_title(f'Label: {labels[i].item()}', fontsize=9)
        ax.axis('off')
    plt.suptitle('Sample Training Batch', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

show_batch(train_loader)

## 4. CNN Model

In [ ]:
class MnistCNN(nn.Module):
    """
    Architecture
    ─────────────────────────────────────────────────
    Input  : (B, 1, 28, 28)
    Block 1: Conv(32) → BN → ReLU → Conv(32) → BN → ReLU → MaxPool → Dropout
    Block 2: Conv(64) → BN → ReLU → Conv(64) → BN → ReLU → MaxPool → Dropout
    Block 3: Conv(128)→ BN → ReLU → AdaptiveAvgPool
    Head   : Flatten → FC(256) → ReLU → Dropout → FC(10)
    """
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1 — 28x28 → 14x14
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(dropout),
            # Block 2 — 14x14 → 7x7
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2), nn.Dropout2d(dropout),
            # Block 3 — 7x7 → 1x1
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = MnistCNN(num_classes=10, dropout=0.3).to(device)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')
print(model)

## 5. Loss, Optimiser & Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)
print('Loss / Optimizer / Scheduler ready ✅')

## 6. Early Stopping & Checkpoint Helpers

In [ ]:
class EarlyStopping:
    """Stops training when val_loss hasn't improved for `patience` epochs."""
    def __init__(self, patience=5, min_delta=1e-4, checkpoint_path='best_model.pth', verbose=True):
        self.patience        = patience
        self.min_delta       = min_delta
        self.checkpoint_path = checkpoint_path
        self.verbose         = verbose
        self.best_loss       = float('inf')
        self.counter         = 0
        self.stop            = False

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            improvement    = self.best_loss - val_loss
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            if self.verbose:
                print(f'  ✅ Val loss improved by {improvement:.6f} → checkpoint saved')
        else:
            self.counter += 1
            if self.verbose:
                print(f'  ⏳ No improvement. Counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.stop = True
                print('  🛑 Early stopping triggered!')


def save_full_checkpoint(epoch, model, optimizer, scheduler, val_loss, path):
    """Save a full training snapshot (resume-capable)."""
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optim_state_dict': optimizer.state_dict(),
        'sched_state_dict': scheduler.state_dict(),
        'val_loss': val_loss,
    }, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    """Restore a training snapshot."""
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer: optimizer.load_state_dict(ckpt['optim_state_dict'])
    if scheduler: scheduler.load_state_dict(ckpt['sched_state_dict'])
    print(f'Checkpoint restored from epoch {ckpt["epoch"]} (val_loss={ckpt["val_loss"]:.4f})')
    return ckpt['epoch']


print('Helpers defined ✅')

## 7. Training & Validation Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += (model(images).argmax(1) == labels).sum().item()
        total        += images.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs        = model(images)
        loss           = criterion(outputs, labels)
        running_loss  += loss.item() * images.size(0)
        correct       += (outputs.argmax(1) == labels).sum().item()
        total         += images.size(0)
    return running_loss / total, correct / total

print('Loop functions defined ✅')

In [ ]:
early_stopping = EarlyStopping(
    patience=ES_PATIENCE, min_delta=ES_MIN_DELTA,
    checkpoint_path=BEST_MODEL_PATH, verbose=True
)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f'{"Epoch":>6} {"Train Loss":>12} {"Train Acc":>10} {"Val Loss":>10} {"Val Acc":>9} {"Time":>8}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    print(f'{epoch:>6} {train_loss:>12.4f} {train_acc:>10.4f} {val_loss:>10.4f} {val_acc:>9.4f} {elapsed:>7.1f}s')

    # Epoch checkpoint — always overwrite 'latest'
    save_full_checkpoint(epoch, model, optimizer, scheduler, val_loss,
                         path=os.path.join(CHECKPOINT_DIR, 'latest.pth'))

    # Early stopping (also saves best weights)
    early_stopping(val_loss, model)
    if early_stopping.stop:
        print(f'\nTraining stopped at epoch {epoch}.')
        break

print('\nTraining complete ✅')

## 8. Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(epochs_ran, history['val_loss'],   label='Val Loss',   linewidth=2, linestyle='--')
best_epoch = int(np.argmin(history['val_loss'])) + 1
ax1.axvline(best_epoch, color='red', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, [a*100 for a in history['train_acc']], label='Train Acc', linewidth=2)
ax2.plot(epochs_ran, [a*100 for a in history['val_acc']],   label='Val Acc',   linewidth=2, linestyle='--')
ax2.axvline(best_epoch, color='red', linestyle=':', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curves'); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Load Best Weights & Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
print(f'Best weights loaded from: {BEST_MODEL_PATH}')

test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'\nTest Loss : {test_loss:.4f}')
print(f'Test Acc  : {test_acc*100:.2f}%')

## 10. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=[str(i) for i in range(10)]))

## 11. Visualise Predictions

In [ ]:
def plot_predictions(model, loader, n=20):
    model.eval()
    images, labels = next(iter(loader))
    images, labels = images[:n], labels[:n]
    with torch.no_grad():
        logits = model(images.to(device))
        probs  = F.softmax(logits, dim=1).cpu()
        preds  = probs.argmax(dim=1)
    cols = 5; rows = n // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 2.8))
    for i, ax in enumerate(axes.flat):
        img   = images[i].squeeze().numpy() * 0.3081 + 0.1307
        true  = labels[i].item()
        pred  = preds[i].item()
        conf  = probs[i, pred].item() * 100
        color = 'green' if pred == true else 'red'
        ax.imshow(img, cmap='gray')
        ax.set_title(f'True:{true}  Pred:{pred}\n{conf:.1f}%', color=color, fontsize=9)
        ax.axis('off')
    plt.suptitle('Predictions (green=correct, red=wrong)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_predictions(model, test_loader, n=20)

## 12. Export Model

In [ ]:
# Weights-only save
torch.save(model.state_dict(), 'mnist_cnn_final.pth')
print('Weights saved → mnist_cnn_final.pth')

# TorchScript export (production-ready)
model.eval()
dummy    = torch.zeros(1, 1, 28, 28).to(device)
scripted = torch.jit.trace(model, dummy)
scripted.save('mnist_cnn_scripted.pt')
print('TorchScript saved → mnist_cnn_scripted.pt')

# Sanity check
loaded = torch.jit.load('mnist_cnn_scripted.pt', map_location=device)
out    = loaded(dummy)
print(f'Output shape: {out.shape}  Predicted class: {out.argmax(dim=1).item()}')